# 남한 지도 생성기 v2 (레고 픽셀아트 소스용)

- **바다**: 파랑 / **육지**: 흰색
- **도(시도) 경계**: 진하게 / **시·군 경계**: 연하게 / **구 경계**: 없음
- **섬 크기 필터**: `ISLAND_MIN_AREA` 이하 섬 제거
- **북한 하단 윤곽** 표시
- 1m × 1.5m 초고해상도 출력

> 픽셀 다운사이징은 사용자가 직접 처리합니다.

In [ ]:
# ── 설정 ──

# 파일 경로
DATA = r"D:\SEMCoWork\Session25_map\skorea-municipalities-2018-geo.json"
NK   = r"D:\SEMCoWork\Session25_map\north_korea.geojson"
OUT  = r"D:\SEMCoWork\Session25_map\south_korea_map.png"

# 출력 크기 (인치)
W_INCH = 10              # 가로 10인치
H_INCH = W_INCH * 1.5     # 세로 1.5m
DPI = 1000                # 인화용 초고해상도

# 섬 필터 (km² 단위)
# 이 면적(km²)보다 작은 섬은 모두 제거
# 2 km² ≈ 작은 섬들만 제거, 200 km² ≈ 거문도(98km²) 제거·제주도(1,848km²) 유지
ISLAND_MIN_AREA_KM2 = 200  # 200 km² 이하 섬 제거
SQDEG_TO_KM2 = 9845  # 1 제곱도 ≈ 9,845 km² (한반도 위도 보정 0.799)

# 색상
SEA_COLOR  = "#4A90D9"   # 바다 (파랑)
LAND_COLOR = "#FFFFFF"   # 육지 (흰색)
NK_COLOR   = "#E8E8E8"   # 북한 (연회색)
SIDO_EDGE  = "#222222"   # 도 경계 (진하게)
CITY_EDGE  = "#888888"   # 시/군 경계 (연하게)
NK_EDGE    = "#555555"   # 북한 윤곽선

# 선 두께 (포인트)
SIDO_LW = 1.5             # 도(시도) 경계선 두께
CITY_LW = 0.5             # 시/군 경계선 두께

# 광역시/특별시 코드
METRO_CODES = {"11", "21", "22", "23", "24", "25", "26"}
SIDO_NAMES = {
    "11": "서울특별시", "21": "부산광역시", "22": "대구광역시", "23": "인천광역시",
    "24": "광주광역시", "25": "대전광역시", "26": "울산광역시", "29": "세종특별자치시",
    "31": "경기도", "32": "강원도", "33": "충청북도", "34": "충청남도",
    "35": "전라북도", "36": "전라남도", "37": "경상북도", "38": "경상남도",
    "39": "제주특별자치도",
}

# 북한 표시 여부 (False면 북한 영역 제외)
SHOW_NK = True

# 북한 절단 위도 (이 위도 아래만 표시)
NK_CUT = 38.71740371170524

# 해안선 단순화 (degree 단위, 0=해제)
# 0.002 ≈ 220m (은은), 0.005 ≈ 550m (보통), 0.01 ≈ 1.1km (강하게)
# 픽셀아트용: 0.005~0.01 추천
COAST_SMOOTH = 0

In [ ]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from shapely.geometry import shape, box, LineString, Polygon
from shapely.ops import unary_union

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


def filter_islands(geom, min_area):
    """MultiPolygon/Polygon에서 min_area 미만 폴리곤(섬) 제거"""
    if geom.is_empty:
        return geom
    if geom.geom_type == "Polygon":
        if geom.area < min_area:
            return None
        return geom
    polys = [p for p in geom.geoms if p.area >= min_area]
    if not polys:
        return None
    return unary_union(polys)


def get_polys(geom):
    """Polygon/MultiPolygon → list[Polygon]"""
    if geom.is_empty:
        return []
    if geom.geom_type == "Polygon":
        return [geom]
    return list(geom.geoms)


# ── 1. 데이터 로드 ──
with open(DATA, encoding="utf-8") as f:
    gj = json.load(f)
features = gj["features"]

sido_groups = {}   # 시도별 폴리곤 (도 경계용)
city_groups = {}   # 시/군별 폴리곤 (시/군 경계용, 구 경계 제거 → 시 단위 병합)

for ft in features:
    name = ft["properties"]["name"]
    code = ft["properties"]["code"]
    if name == "울릉군":
        continue  # 울릉도/독도 제외
    geom = shape(ft["geometry"])
    sido_groups.setdefault(code[:2], []).append(geom)
    if code[:2] in METRO_CODES:
        city = SIDO_NAMES[code[:2]]
    else:
        city = name[:name.index("시") + 1] if "시" in name else name
    city_groups.setdefault(city, []).append(geom)

# ── 2. 병합 + 섬 필터 ──
min_area_sqdeg = ISLAND_MIN_AREA_KM2 / SQDEG_TO_KM2
sido_polys = {}
for code, geoms in sido_groups.items():
    merged = unary_union(geoms)
    filtered = filter_islands(merged, min_area_sqdeg)
    if filtered is not None:
        sido_polys[code] = filtered

city_polys = {}
for city, geoms in city_groups.items():
    merged = unary_union(geoms)
    filtered = filter_islands(merged, min_area_sqdeg)
    if filtered is not None:
        city_polys[city] = filtered

# 전체 남한 육지 (해안선 1회 그리기용)
all_land = unary_union(list(sido_polys.values()))

# 해안선 단순화 (Douglas-Peucker) — 픽셀아트 변환 시 디테일 과다 방지
# draw_map.ipynb 방식: 각 폴리곤을 독립적으로 simplify
if COAST_SMOOTH > 0:
    sido_polys = {k: v.simplify(COAST_SMOOTH, preserve_topology=False) for k, v in sido_polys.items()}
    city_polys = {k: v.simplify(COAST_SMOOTH, preserve_topology=False) for k, v in city_polys.items()}
    all_land = all_land.simplify(COAST_SMOOTH, preserve_topology=False)
    print(f"해안선 단순화 적용: tolerance={COAST_SMOOTH}°")

# ── 3. 북한 데이터 로드 ──
with open(NK, encoding="utf-8") as f:
    nk_gj = json.load(f)
nk_geom = shape(nk_gj["geometry"])
nk_clip = nk_geom.intersection(box(-180, -90, 180, NK_CUT))

print(f"시도: {len(sido_polys)}개, 시/군: {len(city_polys)}개")
print(f"북한 clip: {nk_clip.geom_type}")

In [ ]:
# ── 4. 전체 범위 계산 ──
xs, ys = [], []
for g in sido_polys.values():
    minx, miny, maxx, maxy = g.bounds
    xs += [minx, maxx]
    ys += [miny, maxy]
if SHOW_NK:
    nk_b = nk_clip.bounds
    xs += [nk_b[0], nk_b[2]]
    ys += [nk_b[1], nk_b[3]]
xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)
xpad = (xmax - xmin) * 0.03  # 포항 동쪽 여백
xmax += xpad

# ── 5. 그리기 (draw_map.ipynb 방식: 각 폴리곤 fill+edge → 시도 exterior 두껍게) ──
fig, ax = plt.subplots(figsize=(W_INCH, H_INCH), dpi=DPI)
ax.set_aspect(1.391 / 1.127)  # 실제 지리 비율 보정

# 배경 (바다)
ax.set_facecolor(SEA_COLOR)
fig.set_facecolor(SEA_COLOR)

# 북한 하단 (연회색 채우기)
if SHOW_NK:
    for poly in get_polys(nk_clip):
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=NK_COLOR, edgecolor="none", zorder=1)

# 전체 육지 바탕 (흰색) — 폴리곤 간 미세 틈으로 배경색이 비치는 것 방지
for poly in get_polys(all_land):
    x, y = poly.exterior.xy
    ax.fill(x, y, facecolor=LAND_COLOR, edgecolor="none", zorder=2)

# 시/군 경계선 (연하게) — 바탕 위에 경계선만 그림
for g in city_polys.values():
    for poly in get_polys(g):
        x, y = poly.exterior.xy
        ax.plot(x, y, color=CITY_EDGE, linewidth=CITY_LW, zorder=3)

# 시도 경계 (진하게) — 각 시도 exterior만 두껍게
for g in sido_polys.values():
    for poly in get_polys(g):
        x, y = poly.exterior.xy
        ax.plot(x, y, color=SIDO_EDGE, linewidth=SIDO_LW, zorder=4)

# 북한 윤곽선
if SHOW_NK:
    for poly in get_polys(nk_clip):
        x, y = poly.exterior.xy
        ax.plot(x, y, color=NK_EDGE, linewidth=0.8, zorder=4)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis("off")

plt.tight_layout(pad=0)
plt.savefig(OUT, dpi=DPI, facecolor=fig.get_facecolor())
plt.close(fig)

print("saved:", OUT)
print(f"출력 크기: {int(W_INCH*DPI)} x {int(H_INCH*DPI)} px")
print(f"물리 크기: {W_INCH/39.37:.2f}m x {H_INCH/39.37:.2f}m")
print(f"섬 필터: {ISLAND_MIN_AREA_KM2} km² 이하 제거 (≈{min_area_sqdeg:.6f} 제곱도)")


In [ ]:
# ── 6. 해안선 단순화 강도별 PNG 일괄 생성 ──
# COAST_SMOOTH 값을 순차적으로 증가시켜 픽셀아트용 소스를 비교

import os

# 단순화 강도 후보 (degree 단위)
SMOOTH_LEVELS = [0, 0.002, 0.005, 0.008, 0.01, 0.015, 0.02]

# 출력 디렉토리
OUT_DIR = os.path.join(os.path.dirname(OUT), "smooth_compare")
os.makedirs(OUT_DIR, exist_ok=True)

# 원본(미단순화) 폴리곤을 sido_groups / city_groups에서 재준비
orig_sido = {}
for code, geoms in sido_groups.items():
    merged = unary_union(geoms)
    filtered = filter_islands(merged, min_area_sqdeg)
    if filtered is not None:
        orig_sido[code] = filtered

orig_city = {}
for city, geoms in city_groups.items():
    merged = unary_union(geoms)
    filtered = filter_islands(merged, min_area_sqdeg)
    if filtered is not None:
        orig_city[city] = filtered

orig_all_land = unary_union(list(orig_sido.values()))

def draw_with_smooth(smooth_val, out_path):
    """주어진 smooth 값으로 지도를 그려 out_path에 저장"""
    if smooth_val > 0:
        sp = {k: v.simplify(smooth_val, preserve_topology=False) for k, v in orig_sido.items()}
        cp = {k: v.simplify(smooth_val, preserve_topology=False) for k, v in orig_city.items()}
        al = orig_all_land.simplify(smooth_val, preserve_topology=False)
    else:
        sp, cp, al = orig_sido, orig_city, orig_all_land

    # 범위 계산 (cell 3과 동일)
    xs, ys = [], []
    for g in sp.values():
        minx, miny, maxx, maxy = g.bounds
        xs += [minx, maxx]
        ys += [miny, maxy]
    if SHOW_NK:
        nk_b = nk_clip.bounds
        xs += [nk_b[0], nk_b[2]]
        ys += [nk_b[1], nk_b[3]]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    xpad = (xmax - xmin) * 0.03
    xmax += xpad

    fig, ax = plt.subplots(figsize=(W_INCH, H_INCH), dpi=DPI)
    ax.set_aspect(1.391 / 1.127)

    ax.set_facecolor(SEA_COLOR)
    fig.set_facecolor(SEA_COLOR)

    if SHOW_NK:
        for poly in get_polys(nk_clip):
            x, y = poly.exterior.xy
            ax.fill(x, y, facecolor=NK_COLOR, edgecolor="none", zorder=1)

    # 전체 육지 바탕 (흰색)
    for poly in get_polys(al):
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=LAND_COLOR, edgecolor="none", zorder=2)

    # 시/군 경계선 (연하게)
    for g in cp.values():
        for poly in get_polys(g):
            x, y = poly.exterior.xy
            ax.plot(x, y, color=CITY_EDGE, linewidth=CITY_LW, zorder=3)

    # 시도 경계 (진하게)
    for g in sp.values():
        for poly in get_polys(g):
            x, y = poly.exterior.xy
            ax.plot(x, y, color=SIDO_EDGE, linewidth=SIDO_LW, zorder=4)

    if SHOW_NK:
        for poly in get_polys(nk_clip):
            x, y = poly.exterior.xy
            ax.plot(x, y, color=NK_EDGE, linewidth=0.8, zorder=4)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")

    plt.tight_layout(pad=0)
    plt.savefig(out_path, dpi=DPI, facecolor=fig.get_facecolor())
    plt.close(fig)

for smooth_val in SMOOTH_LEVELS:
    smooth_val_label = f"{smooth_val:.3f}".replace(".", "p")
    out_path = os.path.join(OUT_DIR, f"south_korea_smooth_{smooth_val_label}.png")
    draw_with_smooth(smooth_val, out_path)
    print(f"saved: {out_path}")

print(f"\n완료: {len(SMOOTH_LEVELS)}개 파일 → {OUT_DIR}")
